# Chuẩn bị dữ liệu tóm tắt tiếng Việt



## 1. Kết nối và môi trường


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

drive_root = Path("/content/drive/MyDrive/vietnamese-summarization")
drive_root.mkdir(parents=True, exist_ok=True)

print("Thư mục dự án trên Drive:", drive_root)


In [ ]:
import os
import sys
import subprocess

repo_url = "https://github.com/MD1809/vietnamese-summarization.git"
repo_dir = Path("/content/vietnamese-summarization")
target_commit = ""

if not repo_dir.exists():
    subprocess.run(["git", "clone", repo_url, str(repo_dir)], check=True)

remote_result = subprocess.run(
    ["git", "remote", "get-url", "origin"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True,
)

if remote_result.stdout.strip() != repo_url:
    raise RuntimeError("Lỗi: Repository không đúng dự án.")

status_result = subprocess.run(
    ["git", "status", "--porcelain"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True,
)

if status_result.stdout.strip() != "":
    raise RuntimeError("Lỗi: Repository còn thay đổi. Hãy dùng runtime mới.")

subprocess.run(["git", "fetch", "origin"], cwd=repo_dir, check=True)

if target_commit == "":
    commit_result = subprocess.run(
        ["git", "rev-parse", "origin/main"], cwd=repo_dir,
        capture_output=True, text=True, check=True,
    )
    target_commit = commit_result.stdout.strip()
subprocess.run(
    ["git", "checkout", "--detach", target_commit],
    cwd=repo_dir,
    check=True,
)

os.chdir(repo_dir)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

print("Commit sử dụng:", target_commit)


In [ ]:
%pip uninstall -y -q diffusers torchvision timm fastai
%pip install -q "jedi==0.20.1"
%pip install -q -r /content/vietnamese-summarization/requirements.txt
%pip check


In [ ]:
import json
import src.data
import src.preprocessing
from src.utils import load_train_config

config_path = repo_dir / "configs" / "train.json"
train_config = load_train_config(config_path)

manifest_names = [
    "wikilingua_source.json",
    "wikilingua_cleaning.json",
    "wikilingua_splits.json",
]

reference_manifests = {}

for name in manifest_names:
    manifest_path = repo_dir / "data" / "manifests" / name
    reference_manifests[name] = json.loads(
        manifest_path.read_text(encoding="utf-8")
    )

print(json.dumps(
    {
        "dataset": train_config["dataset"],
        "seed": train_config["seed"],
        "max_source_length": train_config["max_source_length"],
        "max_target_length": train_config["max_target_length"],
    },
    ensure_ascii=False,
    indent=2,
))


## 2. Tải nguồn và chuyển section thành cặp


In [ ]:
cache_dir = drive_root / "data" / "raw"
dataset = src.data.load_wikilingua(cache_dir, config_path)
source_rows = dataset["train"]

expected_article_count = reference_manifests[
    "wikilingua_source.json"
]["splits"]["train"]["article_row_count"]

if len(source_rows) != expected_article_count:
    raise RuntimeError("Lỗi: Số bài nguồn khác manifest đã chốt.")

pairs = src.data.convert_wikilingua_sections(source_rows)

print("Số bài nguồn:", len(source_rows))
print("Số cặp trước làm sạch:", len(pairs))


## 3. Chuẩn hóa và làm sạch


In [ ]:
normalized_pairs = src.preprocessing.normalize_pairs(pairs)

cleaned_pairs, cleaning_report = src.data.clean_wikilingua_pairs(
    normalized_pairs
)

expected_cleaning = reference_manifests["wikilingua_cleaning.json"].copy()
expected_cleaning.pop("created_at_utc", None)
expected_cleaning.pop("dataset", None)

if cleaning_report != expected_cleaning:
    raise RuntimeError("Lỗi: Kết quả làm sạch khác dữ liệu đã chốt.")

print("Số cặp giữ lại:", len(cleaned_pairs))
print("Số cặp bị loại:", cleaning_report["removed_pair_count"])
print("Lý do loại:", cleaning_report["removed_counts_by_reason"])


## 4. Giữ toàn bộ dữ liệu trong các tập đã chia


In [ ]:
dataset_splits, split_manifest = src.data.split_wikilingua_pairs(
    cleaned_pairs,
    cleaning_report,
    config_path,
)

expected_splits = reference_manifests["wikilingua_splits.json"].copy()
expected_splits.pop("created_at_utc", None)

if split_manifest != expected_splits:
    raise RuntimeError("Lỗi: Bộ chia khác manifest đã chốt.")

prepared_splits, data_manifest = src.data.select_full_dataset(
    dataset_splits, split_manifest, config_path,
)

split_counts = {}

for split_name in ["train", "validation", "test"]:
    split_counts[split_name] = len(prepared_splits[split_name])

print(json.dumps(split_counts, ensure_ascii=False, indent=2))


## 5. Thống kê cần thiết

In [ ]:
dataset_statistics = src.data.build_dataset_statistics(
    prepared_splits,
    data_manifest,
    cleaning_report,
    config_path,
)

print(json.dumps(
    {
        "splits": dataset_statistics["splits"],
        "training_token_lengths": dataset_statistics["training_token_lengths"],
    },
    ensure_ascii=False,
    indent=2,
))


## 6. Xuất JSONL và manifest

In [ ]:
# Persist new manifests beside the full data; existing content must match.
manifest_path = drive_root / train_config["paths"]["subset_manifest"]
if manifest_path.exists():
    existing_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if src.data._split_manifest_fingerprint(existing_manifest) != src.data._split_manifest_fingerprint(data_manifest):
        raise RuntimeError("Full dataset manifest differs from the saved selection.")
else:
    src.data.save_subset_manifest(data_manifest, manifest_path)
processed_manifest = src.data.export_prepared_datasets(
    prepared_splits, data_manifest, dataset_statistics, drive_root, config_path,
)

# Measure only train/validation; the test set remains locked for final evaluation.
import src.tokenization
truncation_report = src.tokenization.measure_truncation(drive_root, config_path)
truncation_path = drive_root / train_config["token_limits_decision"]["selected_report"]
src.tokenization.save_truncation_report(truncation_report, truncation_path)

print("Full dataset prepared:", split_counts)
print("Data:", (drive_root / train_config["paths"]["train_file"]).parent)
print("Manifest:", manifest_path)
print("Truncation report:", truncation_path)
print("Test locked:", processed_manifest["test_lock"]["locked"])
